# Module 08 — Notebook 3 Solutions: Comparing Groups

Reference solutions for `03_comparing_groups.ipynb`. Try the exercises yourself first!

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_length, check_contains
import statistics
import random
import math
print("Setup complete.")

## Exercise 1 Solution — Compute Cohen's d

In [ ]:
import statistics
import math

prompt_v1 = [0.72, 0.75, 0.70, 0.74, 0.73, 0.76, 0.71, 0.74, 0.72, 0.75]
prompt_v2 = [0.85, 0.88, 0.82, 0.87, 0.84, 0.89, 0.83, 0.86, 0.85, 0.87]

mean_v1 = statistics.mean(prompt_v1)
mean_v2 = statistics.mean(prompt_v2)
var_v1  = statistics.variance(prompt_v1)
var_v2  = statistics.variance(prompt_v2)
pooled_std = math.sqrt((var_v1 + var_v2) / 2)

# v1 - v2 direction (v1 is group_a)
effect_size = round((mean_v1 - mean_v2) / pooled_std, 4)

print(f"Mean v1: {mean_v1:.4f}")
print(f"Mean v2: {mean_v2:.4f}")
print(f"Cohen's d: {effect_size}")

In [ ]:
check_type(effect_size, float, "effect_size is a float")
check_approx(effect_size, -7.4, 0.5, "effect_size is large and negative (v2 beats v1 by a lot)")

## Exercise 2 Solution — Interpret a P-Value

In [ ]:
p_value = 0.032

# p < 0.05 means statistically significant at the 5% level
is_significant = p_value < 0.05          # True

# When significant, we reject H0 (the null hypothesis of no difference)
reject_null = p_value < 0.05             # True

# p < 0.05 doesn't guarantee the models are different — it means the result
# is unlikely under H0. There's still ~3.2% chance this is a false positive.
means_models_identical = False           # No — statistical significance != proof of difference

print(f"Significant: {is_significant}")
print(f"Reject null: {reject_null}")
print(f"Proves definitively different: {means_models_identical}")

In [ ]:
check_equal(is_significant, True, "p=0.032 is significant at p<0.05")
check_equal(reject_null, True, "we reject the null hypothesis")
check_equal(means_models_identical, False, "p<0.05 doesn't guarantee — it just means unlikely by chance")

## Exercise 3 Solution — Write a Bootstrap CI Function

In [ ]:
import statistics
import random

def bootstrap_mean_ci(data, n_resamples=1000, seed=0):
    """Return 95% bootstrap CI for the mean as (lower, upper) rounded to 4 decimal places."""
    random.seed(seed)
    n = len(data)
    boot_means = []

    for _ in range(n_resamples):
        resample = [random.choice(data) for _ in range(n)]
        boot_means.append(statistics.mean(resample))

    boot_means.sort()

    lower_idx = int(0.025 * n_resamples)
    upper_idx = int(0.975 * n_resamples)

    return (round(boot_means[lower_idx], 4), round(boot_means[upper_idx], 4))


test_scores = [0.82, 0.85, 0.79, 0.88, 0.81, 0.84, 0.80, 0.87, 0.83, 0.86]
ci = bootstrap_mean_ci(test_scores, seed=42)
print(f"95% CI for mean: {ci}")

In [ ]:
check_type(ci, tuple, "bootstrap_mean_ci returns a tuple")
check_length(ci, 2, "CI tuple has 2 elements")
mean_val = statistics.mean(test_scores)
check_equal(ci[0] < mean_val, True, "CI lower is below the mean")
check_equal(ci[1] > mean_val, True, "CI upper is above the mean")
check_equal(ci[0] > 0.78, True, "CI lower is plausible")
check_equal(ci[1] < 0.90, True, "CI upper is plausible")

## Exercise 4 Solution — Full Comparison Report

In [ ]:
import statistics
import math
import random

model_a_scores = [0.88, 0.91, 0.85, 0.93, 0.89, 0.87, 0.92, 0.90, 0.86, 0.94]
model_b_scores = [0.74, 0.71, 0.77, 0.70, 0.73, 0.76, 0.72, 0.75, 0.69, 0.74]

# Cohen's d helper
def cohens_d(group_a, group_b):
    mean_a = statistics.mean(group_a)
    mean_b = statistics.mean(group_b)
    pooled_std = math.sqrt((statistics.variance(group_a) + statistics.variance(group_b)) / 2)
    return (mean_a - mean_b) / pooled_std

# Welch's t helper
def welch_t(group_a, group_b):
    mean_a = statistics.mean(group_a)
    mean_b = statistics.mean(group_b)
    se = math.sqrt(statistics.variance(group_a) / len(group_a) +
                   statistics.variance(group_b) / len(group_b))
    return (mean_a - mean_b) / se

# Bootstrap CI helper
def bootstrap_mean_ci(data, n_resamples=1000, seed=42):
    random.seed(seed)
    n = len(data)
    boot_means = sorted(
        statistics.mean([random.choice(data) for _ in range(n)])
        for _ in range(n_resamples)
    )
    return (round(boot_means[int(0.025 * n_resamples)], 4),
            round(boot_means[int(0.975 * n_resamples)], 4))


comparison = {
    "mean_a":   round(float(statistics.mean(model_a_scores)), 4),
    "mean_b":   round(float(statistics.mean(model_b_scores)), 4),
    "cohens_d": round(cohens_d(model_a_scores, model_b_scores), 4),
    "t_stat":   round(welch_t(model_a_scores, model_b_scores), 4),
    "ci_a":     bootstrap_mean_ci(model_a_scores),
}

print(comparison)

In [ ]:
from src.checks import check_keys, check_approx, check_type
check_keys(comparison, ["mean_a", "mean_b", "cohens_d", "t_stat", "ci_a"], "comparison has correct keys")
check_approx(comparison["mean_a"], 0.895, 0.001, "mean_a is correct")
check_approx(comparison["mean_b"], 0.731, 0.001, "mean_b is correct")
check_equal(comparison["cohens_d"] > 5.0, True, "cohens_d is large (very different groups)")
check_equal(comparison["t_stat"] > 10.0, True, "t_stat is large")
check_type(comparison["ci_a"], tuple, "ci_a is a tuple")